# diffcast — full VAE fine-tune + EDM latent diffusion forecaster (INSAT-3S)

Parallel pipeline to `flowcast/`. Trains the FlowCast AutoencoderKL end-to-end on INSAT data (encoder + decoder both unfrozen), then trains an EDM latent diffusion model on the new latents to predict 48-frame day-ahead forecasts.

**Order of cells (per channel — VIS first, then WV):**

1. Train VAE — Phase 1, target sanity PSNR ≥ 30 dB (~3-6 h on T4)
2. Uncomment `vae.full_ckpt` in the config (sed cell)
3. Sanity check + encode new latents — Phase 2 (~15 min)
4. Train EDM diffusion — Phase 4 (~8-12 h on T4)
5. Forecast with --samples 8 — Phase 5 (~5 min)
6. Inspect metrics + grids

Set runtime to **T4 GPU** (or A100/H100 for faster). `flowcast/` is NOT touched by this notebook.

## Setup — mount Drive, cd, install deps (run after every fresh runtime)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ISRO/ISRO A.1'   # <-- edit if your Drive path differs
%cd "$PROJECT_DIR"
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip install -q lpips==0.1.4 imagecodecs huggingface_hub
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# ── Session status / resume check ───────────────────────────────────
import os, json, datetime as _dt

for ch in ('vis', 'wv'):
    ckpt_dir = f'{ch}/checkpoints_diff'
    out_dir  = f'{ch}/outputs_diff'
    print(f'\n── {ch.upper()} (diffcast) ──────────────────────────────────────────')

    manifest = f'{ch}/manifest_diff.csv'
    print(f'  manifest_diff   : {"✓ " + manifest if os.path.exists(manifest) else "✗ (created by train_vae)"}')

    # Phase 1 — full VAE
    vae_full = os.path.join(ckpt_dir, 'vae_full.pt')
    hist_vae = os.path.join(ckpt_dir, 'vae_full_history.json')
    if os.path.exists(vae_full):
        sz = os.path.getsize(vae_full) / 1e6
        mtime = _dt.datetime.fromtimestamp(os.path.getmtime(vae_full)).strftime('%Y-%m-%d %H:%M')
        best_psnr = '?'
        if os.path.exists(hist_vae):
            h = json.load(open(hist_vae))
            best_psnr = f'{h.get("best_val_psnr", 0):.2f} dB'
            steps_done = len(h.get("history", []))
        else:
            steps_done = '?'
        print(f'  vae_full.pt     : ✓ ({sz:.1f} MB, {mtime}, best PSNR={best_psnr}, {steps_done} val-steps)')
        gate = '✅ PASSED (≥ 30 dB)' if best_psnr != '?' and float(best_psnr.split()[0]) >= 30 else '⚠️  check gate'
        print(f'  Phase 1 gate    : {gate}')
    else:
        print(f'  vae_full.pt     : ✗ (run Phase 1)')

    # Phase 2 — sanity
    sanity = os.path.join(out_dir, 'sanity_metrics.json')
    if os.path.exists(sanity):
        m = json.load(open(sanity))
        psnr = m.get('summary', {}).get('psnr', 0)
        gate_flag = m.get('passed_gate', psnr >= 30)
        print(f'  sanity_check    : ✓ PSNR={psnr:.2f} dB  gate={"✅" if gate_flag else "⚠️  FAILED"}')
    else:
        print(f'  sanity_check    : ✗ (run Phase 2)')

    # Phase 4 — diffusion model
    best_diff = os.path.join(ckpt_dir, 'best_diff.pt')
    hist_diff = os.path.join(ckpt_dir, 'diff_history.json')
    if os.path.exists(best_diff):
        sz = os.path.getsize(best_diff) / 1e6
        mtime = _dt.datetime.fromtimestamp(os.path.getmtime(best_diff)).strftime('%Y-%m-%d %H:%M')
        epochs_done = len(json.load(open(hist_diff)).get('history', [])) if os.path.exists(hist_diff) else '?'
        print(f'  best_diff.pt    : ✓ ({sz:.1f} MB, {mtime}, {epochs_done} epochs)')
    else:
        print(f'  best_diff.pt    : ✗ (run Phase 4)')

    # Phase 5 — forecast outputs
    if os.path.isdir(out_dir):
        fc_files = [f for f in os.listdir(out_dir) if 'diff_forecast' in f and f.endswith('_metrics.json')]
        print(f'  forecast outputs: {len(fc_files)} metrics file(s): {fc_files or "(none)"}')
print()


## VIS — Phase 1: full VAE fine-tune (encoder + decoder)

Loads SEVIR pretrained → unfreezes everything → trains 15,000 steps with `1.0·MSE + 0.5·LPIPS-VGG + 1e-6·KL`. Saves best-by-val-PSNR to `vis/checkpoints_diff/vae_full.pt`. Progress PNGs at `vis/outputs_diff/vae_train_progress/`.

**Gate: val PSNR ≥ 30 dB.** If below, extend with `--steps 25000`.

In [ ]:
!python -m diffcast.train_vae --config vis/config_diff.yaml

In [ ]:
# ── Phase 1: VAE training history plot ─────────────────────────────
import json, os
import matplotlib.pyplot as plt

hist_path = 'vis/checkpoints_diff/vae_full_history.json'
if not os.path.exists(hist_path):
    print('No history yet — run Phase 1 (train_vae) first.')
else:
    h = json.load(open(hist_path))
    history  = h.get('history', [])
    baseline = h.get('baseline', {})
    best     = h.get('best_val_psnr', 0)

    steps  = [e['step']     for e in history]
    psnrs  = [e['val_psnr'] for e in history]
    losses = [e['loss']     for e in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(steps, psnrs, label='val PSNR', color='steelblue', linewidth=2)
    ax1.axhline(baseline.get('val_psnr', 0), color='gray',   linestyle='--',
                label=f'baseline (SEVIR) = {baseline.get("val_psnr", 0):.1f} dB', alpha=0.7)
    ax1.axhline(30, color='green', linestyle=':', label='gate = 30 dB', alpha=0.8)
    ax1.axhline(best, color='orange', linestyle='--', alpha=0.7,
                label=f'best = {best:.2f} dB')
    ax1.set_xlabel('Step'); ax1.set_ylabel('Val PSNR (dB)')
    ax1.set_title('DiffCast VIS VAE — Val PSNR')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(steps, losses, label='total loss', color='tomato', linewidth=2)
    ax2.set_xlabel('Step'); ax2.set_ylabel('Loss')
    ax2.set_title('DiffCast VIS VAE — Training loss')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f'Steps completed : {steps[-1] if steps else 0}')
    print(f'Best val PSNR   : {best:.2f} dB  (gate: 30 dB)')
    print(f'Baseline PSNR   : {baseline.get("val_psnr", 0):.2f} dB  (frozen SEVIR)')
    gate_label = '✅  GATE PASSED' if best >= 30 else f'⚠️   GATE NOT YET PASSED ({best:.2f} dB < 30 dB — run extension cell below)'
    print(gate_label)


In [ ]:
# ── Phase 1: VAE training progress images ──────────────────────────
# Shows decoded vs GT comparison grids saved every 500 training steps.
import glob, os
from IPython.display import Image, display

prog_dir = 'vis/outputs_diff/vae_train_progress'
imgs = sorted(glob.glob(os.path.join(prog_dir, '*.png')))
if not imgs:
    print('No progress images yet — run Phase 1 first.')
else:
    show = ([imgs[0]] + imgs[len(imgs)//2:len(imgs)//2+1] + imgs[-2:]) if len(imgs) > 3 else imgs
    for p in show:
        print(p)
        display(Image(p))


In [ ]:
# ── Phase 1 gate check + optional extension ────────────────────────
import json, os

hist_path = 'vis/checkpoints_diff/vae_full_history.json'
if not os.path.exists(hist_path):
    raise FileNotFoundError('Run Phase 1 (train_vae) first.')

h = json.load(open(hist_path))
best_psnr    = h.get('best_val_psnr', 0)
target_psnr  = h.get('target_psnr', 30.0)
passed       = best_psnr >= target_psnr

print(f'Best val PSNR : {best_psnr:.2f} dB   Target: {target_psnr:.1f} dB')
if passed:
    print('✅  GATE PASSED — proceed to Phase 2 (enable VAE + sanity_check + encode_latents).')
else:
    print(f'⚠️   GATE NOT PASSED ({best_psnr:.2f} dB < {target_psnr:.1f} dB).')
    print('    Recommendation: extend training with --steps 25000.')
    print('    Check the loss curves above — if PSNR is still rising, more steps will help.')
    print()
    print('    To extend, un-comment and run the cell below:')
    print()
    print('      !python -m diffcast.train_vae --config vis/config_diff.yaml --steps 25000')


In [ ]:
# ── Extension cell: run only if Phase 1 gate FAILED ─────────────────
# Un-comment the line below and run this cell.
# The script starts from scratch but overwrites best checkpoint only on improvement.

# !python -m diffcast.train_vae --config vis/config_diff.yaml --steps 25000


## VIS — Phase 2: enable the fine-tuned VAE in config + sanity check + encode latents

Uncomments `vae.full_ckpt: vis/checkpoints_diff/vae_full.pt` in the config, runs sanity_check (gate ≥ 30 dB), then encodes all VIS TIFFs to NEW latents at `vis/latents_diff/`.

In [ ]:
# Enable the fine-tuned VAE for inference
import re, pathlib
cfg_path = pathlib.Path('vis/config_diff.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)#\s*full_ckpt:\s*vis/checkpoints_diff/vae_full\.pt.*$',
    r'\1full_ckpt: vis/checkpoints_diff/vae_full.pt',
    text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
for line in new_text.splitlines():
    if 'vae' in line.lower() or 'full_ckpt' in line:
        print(' ', line)

In [ ]:
!python -m diffcast.sanity_check --config vis/config_diff.yaml --num 30
!python -m diffcast.encode_latents --config vis/config_diff.yaml

## VIS — Phase 4: train EDM latent diffusion forecaster (~8-12 h on T4)

Same Earthformer-UNet backbone flowcast uses, wrapped with EDM preconditioning + Karras-schedule sampling. 100 epochs, fp16, EMA. Best-EMA checkpoint at `vis/checkpoints_diff/best_diff.pt`.

In [ ]:
!python -m diffcast.train_diffusion --config vis/config_diff.yaml

In [ ]:
# ── Phase 4: diffusion training history plot ────────────────────────
import json, os
import matplotlib.pyplot as plt

hist_path = 'vis/checkpoints_diff/diff_history.json'
if not os.path.exists(hist_path):
    print('No history yet — run Phase 4 (train_diffusion) first.')
else:
    h = json.load(open(hist_path))
    history  = h.get('history', [])
    best_val = h.get('best_val', min(e['val_loss'] for e in history) if history else 0)

    epochs  = [e['epoch']      for e in history]
    tr_loss = [e['train_loss'] for e in history]
    va_loss = [e['val_loss']   for e in history]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(epochs, tr_loss, label='train EDM loss')
    ax.plot(epochs, va_loss, label='val EDM loss', linewidth=2)
    ax.axhline(best_val, color='green', linestyle='--', alpha=0.7,
               label=f'best val = {best_val:.5f}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('EDM Loss (σ-weighted denoising)')
    ax.set_title('DiffCast VIS — Diffusion model training curve')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f'Epochs completed: {len(history)}   Best val loss: {best_val:.5f}')
    if history:
        last = history[-1]
        print(f'Last epoch {last["epoch"]}: train={last["train_loss"]:.5f}  val={last["val_loss"]:.5f}')


## VIS — Phase 5: 48-frame forecast (3-way ensemble: pixmean / medoid / latmean)

Heun ODE sampling (18 steps per autoregressive block × 4 blocks = 48 frames), 8-sample ensemble, decoded through the fine-tuned VAE.

In [ ]:
!python -m diffcast.forecast_diff --config vis/config_diff.yaml --samples 8

In [ ]:
# Display metrics + grids
import glob, json
from IPython.display import Image, display

for p in sorted(glob.glob('vis/outputs_diff/diff_forecast_*_grid.png'))[-3:]:
    print(p); display(Image(p))

for p in sorted(glob.glob('vis/outputs_diff/diff_forecast_*_metrics.json')):
    m = json.load(open(p))
    print(f"\n{p}  samples={m.get('samples')}  medoid_idx={m.get('medoid_idx')}  "
          f"sampler_steps={m.get('sampler_steps')}")
    print(f"  thresholds: {m.get('thresholds')}")
    for mode, smry in (m.get('summary_by_mode') or {}).items():
        print(f"  [{mode}]")
        for k, v in (smry or {}).items():
            print(f"    {k:>10}: {v:.4f}")
    if m.get('crps_mean') is not None:
        print(f"  [crps]\n    CRPS: {m['crps_mean']:.4f}")

## VIS — Phase 6: comparison vs flowcast

Side-by-side: VIS diffcast vs VIS flowcast (baseline / A / B). Same forecast date, same metrics.

In [ ]:
# Compare diffcast vs flowcast on the same forecast date
import glob, json, os

print(f"{'pipeline':<30s} {'mode':<10s} {'psnr':>7s} {'ssim':>7s} {'CSI_M':>7s}  {'CRPS':>8s}")
print('-' * 75)

# flowcast metrics (all live in vis/outputs_fc/)
for p in sorted(glob.glob('vis/outputs_fc/flow_forecast_*_ens8_metrics*.json')):
    m = json.load(open(p))
    label = ('flowcast/' + os.path.basename(p)
             .replace('flow_forecast_', '')
             .replace('_metrics.json', '')
             .replace('_metrics_', '_'))[:30]
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        psnr = smry.get('psnr', 0); ssim = smry.get('ssim', 0); csi = smry.get('CSI_M', 0)
        crps = m.get('crps_mean')
        crps_str = f"{crps:.4f}" if crps is not None else 'n/a'
        print(f"{label:<30s} {mode:<10s} {psnr:7.3f} {ssim:7.3f} {csi:7.3f}  {crps_str:>8s}")

# diffcast metrics
for p in sorted(glob.glob('vis/outputs_diff/diff_forecast_*_ens8_metrics.json')):
    m = json.load(open(p))
    label = ('diffcast/' + os.path.basename(p)
             .replace('diff_forecast_', '')
             .replace('_metrics.json', ''))[:30]
    for mode, smry in (m.get('summary_by_mode') or {}).items():
        psnr = smry.get('psnr', 0); ssim = smry.get('ssim', 0); csi = smry.get('CSI_M', 0)
        crps = m.get('crps_mean')
        crps_str = f"{crps:.4f}" if crps is not None else 'n/a'
        print(f"{label:<30s} {mode:<10s} {psnr:7.3f} {ssim:7.3f} {csi:7.3f}  {crps_str:>8s}")

## WV — Phases 1 → 5 (same recipe, swap config)

Run only after VIS validates. WV is 24h continuous so no day/night handling; expect smaller gains than VIS (the SEVIR VAE was already in-distribution for WV per the flowcast experiments).

In [ ]:
!python -m diffcast.train_vae --config wv/config_diff.yaml

In [ ]:
# ── WV Phase 1: VAE training history plot ────────────────────────
import json, os
import matplotlib.pyplot as plt

hist_path = 'wv/checkpoints_diff/vae_full_history.json'
if not os.path.exists(hist_path):
    print('No history yet — run WV Phase 1 first.')
else:
    h = json.load(open(hist_path))
    history = h.get('history', [])
    baseline = h.get('baseline', {})
    best = h.get('best_val_psnr', 0)
    steps  = [e['step']     for e in history]
    psnrs  = [e['val_psnr'] for e in history]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, psnrs, label='val PSNR', linewidth=2)
    ax.axhline(baseline.get('val_psnr', 0), color='gray', linestyle='--',
               label=f'baseline = {baseline.get("val_psnr", 0):.1f} dB', alpha=0.7)
    ax.axhline(30, color='green', linestyle=':', label='gate = 30 dB', alpha=0.8)
    ax.set_xlabel('Step'); ax.set_ylabel('Val PSNR (dB)')
    ax.set_title('DiffCast WV VAE — Val PSNR')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f'Best val PSNR: {best:.2f} dB  (gate: 30 dB)')
    print('✅  GATE PASSED' if best >= 30 else f'⚠️   GATE NOT YET PASSED — consider --steps 25000')


In [ ]:
# ── WV Phase 1: VAE training progress images ─────────────────────
import glob, os
from IPython.display import Image, display

prog_dir = 'wv/outputs_diff/vae_train_progress'
imgs = sorted(glob.glob(os.path.join(prog_dir, '*.png')))
if not imgs:
    print('No progress images yet.')
else:
    show = ([imgs[0]] + imgs[-2:]) if len(imgs) > 2 else imgs
    for p in show: print(p); display(Image(p))


In [ ]:
# ── WV Phase 1 gate check ────────────────────────────────────────
import json, os

hist_path = 'wv/checkpoints_diff/vae_full_history.json'
if not os.path.exists(hist_path):
    raise FileNotFoundError('Run WV Phase 1 (train_vae) first.')
h = json.load(open(hist_path))
best_psnr = h.get('best_val_psnr', 0)
print(f'WV best val PSNR : {best_psnr:.2f} dB   Target: 30.0 dB')
if best_psnr >= 30:
    print('✅  WV gate PASSED — proceed to Phase 2.')
else:
    print('⚠️   WV gate not passed. Extend with:')
    print('    !python -m diffcast.train_vae --config wv/config_diff.yaml --steps 25000')


In [ ]:
import re, pathlib
cfg_path = pathlib.Path('wv/config_diff.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)#\s*full_ckpt:\s*wv/checkpoints_diff/vae_full\.pt.*$',
    r'\1full_ckpt: wv/checkpoints_diff/vae_full.pt',
    text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
for line in new_text.splitlines():
    if 'full_ckpt' in line: print(' ', line)

In [ ]:
!python -m diffcast.sanity_check --config wv/config_diff.yaml --num 30
!python -m diffcast.encode_latents --config wv/config_diff.yaml

In [ ]:
!python -m diffcast.train_diffusion --config wv/config_diff.yaml

In [ ]:
# ── WV Phase 4: diffusion training history ───────────────────────
import json, os
import matplotlib.pyplot as plt

hist_path = 'wv/checkpoints_diff/diff_history.json'
if not os.path.exists(hist_path):
    print('No history yet.')
else:
    h = json.load(open(hist_path))
    history = h.get('history', [])
    best_val = h.get('best_val', min(e['val_loss'] for e in history) if history else 0)
    epochs  = [e['epoch']      for e in history]
    tr_loss = [e['train_loss'] for e in history]
    va_loss = [e['val_loss']   for e in history]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(epochs, tr_loss, label='train'); ax.plot(epochs, va_loss, label='val', lw=2)
    ax.axhline(best_val, color='green', ls='--', alpha=0.7, label=f'best={best_val:.5f}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('EDM Loss'); ax.set_title('DiffCast WV — Diffusion training')
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    print(f'Epochs done: {len(history)}  Best val: {best_val:.5f}')


In [ ]:
!python -m diffcast.forecast_diff --config wv/config_diff.yaml --samples 8

In [ ]:
import glob, json
from IPython.display import Image, display
for p in sorted(glob.glob('wv/outputs_diff/diff_forecast_*_grid.png'))[-3:]:
    print(p); display(Image(p))
for p in sorted(glob.glob('wv/outputs_diff/diff_forecast_*_metrics.json')):
    m = json.load(open(p))
    print(f"\n{p}  samples={m.get('samples')}")
    for mode, smry in (m.get('summary_by_mode') or {}).items():
        print(f"  [{mode}]")
        for k, v in (smry or {}).items():
            print(f"    {k:>10}: {v:.4f}")
    if m.get('crps_mean') is not None:
        print(f"  [crps] CRPS: {m['crps_mean']:.4f}")

## Snapshot (both channels)

Bundles all diffcast outputs for download. Mirrors the flowcast snapshot pattern; uses `_diff` suffix so it doesn't collide with existing flowcast snapshots.

In [ ]:
import os, shutil, glob, json, zipfile, datetime as dt
import yaml

TS = dt.datetime.now().strftime('%Y%m%d_%H%M%S')

def snapshot_channel(CH):
    SNAP = f'{CH}/results_snapshot_diff_{TS}'
    os.makedirs(SNAP, exist_ok=True)
    for p in glob.glob(f'{CH}/outputs_diff/*'):
        if os.path.isfile(p):
            shutil.copy2(p, os.path.join(SNAP, os.path.basename(p)))
    if os.path.isdir(f'{CH}/outputs_diff/vae_train_progress'):
        shutil.copytree(f'{CH}/outputs_diff/vae_train_progress',
                        os.path.join(SNAP, 'vae_train_progress'),
                        dirs_exist_ok=True)
    for src in glob.glob(f'{CH}/checkpoints_diff/*.pt') + \
                glob.glob(f'{CH}/checkpoints_diff/*.json'):
        shutil.copy2(src, os.path.join(SNAP, os.path.basename(src)))
    shutil.copy2(f'{CH}/config_diff.yaml', os.path.join(SNAP, 'config_diff.yaml'))

    zip_path = f'{CH}/results_snapshot_diff_{TS}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(SNAP):
            for f in files:
                full = os.path.join(root, f)
                arc = os.path.relpath(full, os.path.dirname(SNAP))
                zf.write(full, arcname=arc)
    sz = os.path.getsize(zip_path) / (1024 * 1024)
    print(f'[{CH}] snapshot: {SNAP}  zip: {zip_path}  ({sz:.1f} MB)')

for CH in ('vis', 'wv'):
    if os.path.isdir(f'{CH}/outputs_diff'):
        snapshot_channel(CH)
    else:
        print(f'[{CH}] skipped — no outputs_diff (channel not yet run)')